# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

# Se incluye el modulo para que este repositorio sea ejecutable por separado.
Path("cleaning.py").write_text("import pandas as pd\n\ndef clean_column_names(data):\n    result = data.copy()\n    result.columns = result.columns.str.strip().str.lower().str.replace(r\"\\s+\", \"_\", regex=True)\n    return result.rename(columns={\"st\": \"state\"})\n\ndef clean_invalid_values(data):\n    result = data.copy()\n    for column in result.select_dtypes(include=[\"object\", \"string\"]):\n        result[column] = result[column].astype(\"string\").str.strip()\n    result[\"gender\"] = result[\"gender\"].str.lower().replace({\"f\": \"F\", \"female\": \"F\", \"femal\": \"F\", \"m\": \"M\", \"male\": \"M\"})\n    result[\"state\"] = result[\"state\"].replace({\"AZ\": \"Arizona\", \"Cali\": \"California\", \"CA\": \"California\", \"WA\": \"Washington\"})\n    result[\"education\"] = result[\"education\"].replace({\"Bachelors\": \"Bachelor\"})\n    result[\"customer_lifetime_value\"] = result[\"customer_lifetime_value\"].str.replace(\"%\", \"\", regex=False)\n    result[\"vehicle_class\"] = result[\"vehicle_class\"].replace({\"Sports Car\": \"Luxury\", \"Luxury SUV\": \"Luxury\", \"Luxury Car\": \"Luxury\"})\n    return result\n\ndef format_types(data):\n    result = data.copy()\n    complaints = result[\"number_of_open_complaints\"].astype(\"string\")\n    # Los valores 1/5/00 representan cinco quejas, no una fecha.\n    complaints = complaints.map(lambda value: value.split(\"/\")[1] if pd.notna(value) and \"/\" in value else value)\n    result[\"number_of_open_complaints\"] = pd.to_numeric(complaints, errors=\"raise\").astype(\"Int64\")\n    for column in [\"customer_lifetime_value\", \"income\", \"monthly_premium_auto\", \"total_claim_amount\"]:\n        result[column] = pd.to_numeric(result[column], errors=\"raise\")\n    return result\n\ndef handle_nulls(data):\n    # No inventar clientes a partir de filas vacias o sin identificador.\n    result = data.dropna(how=\"all\").dropna(subset=[\"customer\"]).copy()\n    # Deduplicar antes de calcular medianas evita ponderar registros repetidos.\n    reference = result.drop_duplicates()\n    numeric = result.select_dtypes(include=\"number\").columns\n    for column in numeric:\n        result[column] = result[column].fillna(reference[column].median())\n    # Unknown conserva la incertidumbre: no inferimos genero ni estado por la moda.\n    for column in result.select_dtypes(include=[\"object\", \"string\"]):\n        result[column] = result[column].fillna(\"Unknown\")\n    # El enunciado pide enteros: redondeamos; se pierde precision monetaria.\n    result[numeric] = result[numeric].round().astype(\"int64\")\n    return result\n\ndef remove_duplicates(data):\n    # Solo duplicados exactos; dos registros distintos del mismo cliente se conservan.\n    return data.drop_duplicates(keep=\"first\").reset_index(drop=True)\n\ndef clean_data(data):\n    result = clean_column_names(data)\n    result = clean_invalid_values(result)\n    result = format_types(result)\n    result = handle_nulls(result)\n    return remove_duplicates(result)\n", encoding="utf-8")
from cleaning import clean_column_names, clean_invalid_values, format_types, handle_nulls, remove_duplicates

base = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/"
frames = []
for name in ["file1.csv", "file2.csv", "file3.csv"]:
    part = clean_column_names(pd.read_csv(base + name))
    # Primero igualar nombres: ST y State no deben crear dos columnas.
    # Los CLV con % estan expresados a escala 100 frente al file3 numerico.
    text_clv = part["customer_lifetime_value"].astype("string")
    percent_mask = text_clv.str.endswith("%", na=False)
    part = clean_invalid_values(part)
    part = format_types(part)
    part.loc[percent_mask, "customer_lifetime_value"] = part.loc[percent_mask, "customer_lifetime_value"] / 100
    frames.append(part)

combined_raw = pd.concat(frames, ignore_index=True)
print("Filas combinadas antes de limpiar nulos:", len(combined_raw))
combined = remove_duplicates(handle_nulls(combined_raw))
assert not combined.isna().any().any()
assert not combined.duplicated().any()
combined.to_csv("customers_combined_cleaned.csv", index=False)
display(combined.head())
print(combined.shape)

# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [ ]:
marketing = pd.read_csv(base + "marketing_customer_analysis_clean.csv")
marketing = clean_column_names(marketing)
# Eliminar solo columnas de indice exportadas, sin borrar variables de negocio.
marketing = marketing.loc[:, ~marketing.columns.str.startswith("unnamed")].copy()
marketing["effective_to_date"] = pd.to_datetime(marketing["effective_to_date"], format="%Y-%m-%d", errors="raise")
marketing["month"] = marketing["effective_to_date"].dt.month
print(marketing.shape)
display(marketing.head())

1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

In [ ]:
# No existe revenue. Usamos prima mensual como aproximacion explicita.
# Income corresponde a los ingresos del cliente y total_claim_amount a siniestros.
revenue_by_channel = marketing.pivot_table(index="sales_channel", values="monthly_premium_auto", aggfunc="sum").rename(columns={"monthly_premium_auto": "total_monthly_premium"}).round(2).sort_values("total_monthly_premium", ascending=False)
display(revenue_by_channel)
print("Canal con mayor suma de primas mensuales:", revenue_by_channel.index[0])
print("No equivale a ingresos contables ni permite comparar conversion sin conocer exposicion y costes por canal.")

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

In [ ]:
clv_by_gender_education = marketing.pivot_table(index="education", columns="gender", values="customer_lifetime_value", aggfunc="mean").round(2)
segment_sizes = marketing.pivot_table(index="education", columns="gender", values="customer_lifetime_value", aggfunc="count")
display(clv_by_gender_education)
display(segment_sizes)
best_segment = clv_by_gender_education.stack().idxmax()
print("Mayor CLV medio (educacion, genero):", best_segment)
print("Diferencias descriptivas: revisar tamanos de muestra; no demuestran causalidad.")

## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [ ]:
# Se suman quejas, no se cuentan filas. Algunos valores ya estan imputados.
complaints_wide = marketing.pivot_table(index="policy_type", columns="month", values="number_of_open_complaints", aggfunc="sum", fill_value=0)
complaints_long = complaints_wide.reset_index().melt(id_vars="policy_type", var_name="month", value_name="open_complaints").sort_values(["policy_type", "month"]).reset_index(drop=True)
display(complaints_long)
max_per_policy = complaints_long.loc[complaints_long["open_complaints"].eq(complaints_long.groupby("policy_type")["open_complaints"].transform("max"))]
display(max_per_policy)
print("Las sumas pueden ser fraccionarias por imputacion previa. El mes es de vigencia de poliza, no fecha de la queja.")

Solucion preparada con asistencia de IA. Requiere ejecutar el cuaderno; no se han inventado resultados. Para combinar los tres CSV se homogeneiza la escala del CLV: los valores con % se dividen entre 100; el primer laboratorio solo exigia eliminar el simbolo. La interpretacion monetaria debe confirmarse con el propietario de los datos.